# Movie Recommendation

Create movie recommendation with all the fancy techniques of the hybrid collection

### Initialize Client and Collection

In [1]:
import os

from qdrant_client import QdrantClient

client = QdrantClient(
    url=os.getenv('QDRANT_URL'),
    api_key=os.getenv('QDRANT_API_KEY')
)

global collection_name
collection_name = 'recommendation_movie'

client.get_collections()



/Users/andrei/projects/qdrant-essesials/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/pq/gsx7ryn905s1bypf32zs4k980000gn/T/ipykernel_83978/609223453.py:5: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


CollectionsResponse(collections=[CollectionDescription(name='midjourney'), CollectionDescription(name='quantized_binary_2bit_wolf-food-db'), CollectionDescription(name='quantized_binary_wolf-food-db'), CollectionDescription(name='quantized_scalar_wolf-food-db'), CollectionDescription(name='recommendation_movie'), CollectionDescription(name='sport-drug'), CollectionDescription(name='wolf-food-db')])

In [35]:
from qdrant_client.models import (
    Distance,
    HnswConfigDiff,
    MultiVectorComparator,
    MultiVectorConfig,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
)

vectors_config = {
    'dense': VectorParams(
        size=384, 
        distance=Distance.COSINE
    ),
    'colbert': VectorParams(
        size=128,
        distance=Distance.COSINE,
        multivector_config=MultiVectorConfig(
            comparator=MultiVectorComparator.MAX_SIM,
        ),
        hnsw_config=HnswConfigDiff(
            m=0
        )
    )
}

sparse_config = {
    'sparse': SparseVectorParams(
        index=SparseIndexParams(on_disk=False)
    )
}

In [135]:
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name,
    vectors_config=vectors_config,
    sparse_vectors_config=sparse_config
)

client.get_collection(collection_name)

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=0, segments_count=3, config=CollectionConfig(params=CollectionParams(vectors={'colbert': VectorParams(size=128, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=HnswConfigDiff(m=0, ef_construct=None, full_scan_threshold=None, max_indexing_threads=None, on_disk=None, payload_m=None, inline_storage=None), quantization_config=None, on_disk=None, datatype=None, multivector_config=MultiVectorConfig(comparator=<MultiVectorComparator.MAX_SIM: 'max_sim'>)), 'dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors={'sparse': SparseVectorParams(index

##### Create payload indices

In [136]:
from qdrant_client.models import PayloadSchemaType

client.create_payload_index(
    collection_name=collection_name,
    field_name='category',
    field_schema=PayloadSchemaType.KEYWORD
)

client.create_payload_index(
    collection_name=collection_name,
    field_name='release_date',
    field_schema=PayloadSchemaType.DATETIME
)

client.create_payload_index(
    collection_name=collection_name,
    field_name='genre',
    field_schema=PayloadSchemaType.KEYWORD
)

client.create_payload_index(
    collection_name=collection_name,
    field_name='popularity',
    field_schema=PayloadSchemaType.FLOAT
)

client.create_payload_index(
    collection_name=collection_name,
    field_name='vote_average',
    field_schema=PayloadSchemaType.FLOAT
)

UpdateResult(operation_id=10, status=<UpdateStatus.COMPLETED: 'completed'>)

In [5]:
from datasets import load_dataset

ds = load_dataset("Pablinho/movies-dataset")

In [6]:
movies = ds['train']

movies

Dataset({
    features: ['Release_Date', 'Title', 'Overview', 'Popularity', 'Vote_Count', 'Vote_Average', 'Original_Language', 'Genre', 'Poster_Url'],
    num_rows: 9837
})

In [7]:
vars(movies[0])

TypeError: vars() argument must have __dict__ attribute

In [8]:
import re 
def to_snake_case(name):
    name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
    name = re.sub(r'[\s\-]+', '_', name)
    return name.lower()


rename_map = {
    col: to_snake_case(col)
    for col in movies.column_names
}

rename_map

{'Release_Date': 'release_date',
 'Title': 'title',
 'Overview': 'overview',
 'Popularity': 'popularity',
 'Vote_Count': 'vote_count',
 'Vote_Average': 'vote_average',
 'Original_Language': 'original_language',
 'Genre': 'genre',
 'Poster_Url': 'poster_url'}

### Fixing dataset types

In [9]:
movies = movies.rename_columns(rename_map)

movies

Dataset({
    features: ['release_date', 'title', 'overview', 'popularity', 'vote_count', 'vote_average', 'original_language', 'genre', 'poster_url'],
    num_rows: 9837
})

In [85]:
movies['vote_count']

Column(['8940', '1151', '122', '5076', '1793', ...])

In [ ]:
from typing import Any


def clean_vote_count(movie: dict[str, Any]):
    try:
        return {"vote_count": int(movie["vote_count"])}
    except (TypeError, ValueError):
        return {"vote_count": 0}

movies = movies.map(clean_vote_count)

movies['vote_count']

Map: 100%|██████████| 9837/9837 [00:00<00:00, 51951.05 examples/s]


Dataset({
    features: ['release_date', 'title', 'overview', 'popularity', 'vote_count', 'vote_average', 'original_language', 'genre', 'poster_url'],
    num_rows: 9837
})

In [125]:
float(ds['train'][0]['Vote_Average'])

8.3

In [127]:
def clean_vote_average(movie: dict[str, Any], idx: int):
    try:
        return {"vote_average": float(ds['train'][idx]['Vote_Average'])}
    except (TypeError, ValueError):
        return {"vote_average": 0}

movies = movies.map(clean_vote_average, with_indices=True)

type(movies['vote_average'][0])

Map: 100%|██████████| 9837/9837 [00:00<00:00, 24944.24 examples/s]


float

In [128]:
movies['vote_average'][0]

8.3

In [94]:
def genre_split(movie: dict[str, Any]):
    try:
        return {"genre": movie['genre'].lower().split(", ")}
    except (TypeError, ValueError, AttributeError):
        return {"genre": []}

movies = movies.map(genre_split)

movies['genre'][0]

Map: 100%|██████████| 9837/9837 [00:00<00:00, 50599.72 examples/s]


['action', 'adventure', 'science fiction']

In [95]:
def release_to_date(movie: dict[str, Any]):
    try:
        return {"release_date": datetime.strptime(movie['release_date'], "%Y-%m-%d")}
    except (TypeError, ValueError, AttributeError):
        return {"release_date": None}

movies = movies.map(release_to_date)

movies['release_date'][0]

Map: 100%|██████████| 9837/9837 [00:00<00:00, 43799.38 examples/s]


datetime.datetime(2021, 12, 15, 0, 0)

In [96]:
movies.add_column('category', ['movie'] * len(movies))

Dataset({
    features: ['release_date', 'title', 'overview', 'popularity', 'vote_count', 'vote_average', 'original_language', 'genre', 'poster_url', 'category'],
    num_rows: 9837
})

In [97]:
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding, TextEmbedding

DENSE_MODEL_ID = 'sentence-transformers/all-MiniLM-L6-v2' # 384
SPARSE_MODEL_ID = "prithivida/Splade_PP_en_v1"  # SPLADE sparse
COLBERT_MODEL_ID = "colbert-ir/colbertv2.0"  # 128-dim multivector

In [98]:


dense_model = TextEmbedding(DENSE_MODEL_ID)
sparse_model = SparseTextEmbedding(SPARSE_MODEL_ID)
colbert_model = LateInteractionTextEmbedding(COLBERT_MODEL_ID)


In [101]:
import math

texts = [movie['overview'] for movie in movies if isinstance(movie, dict)]

docs = [str(d) for d in texts if d is not None and not (isinstance(d, float) and math.isnan(d))]

In [102]:
len(docs)

9828

In [103]:
docs_idx = [idx for idx, d in enumerate(texts) if d is not None and not (isinstance(d, float) and math.isnan(d))]

docs_idx[-30:]

[9807,
 9808,
 9809,
 9810,
 9811,
 9812,
 9813,
 9814,
 9815,
 9816,
 9817,
 9818,
 9819,
 9820,
 9821,
 9822,
 9823,
 9824,
 9825,
 9826,
 9827,
 9828,
 9829,
 9830,
 9831,
 9832,
 9833,
 9834,
 9835,
 9836]

In [104]:
len(docs), len(texts)

(9828, 9837)

In [105]:
dense_embeds = list(dense_model.embed(docs))

In [106]:
sparse_embeds =  list(sparse_model.embed(docs))
colbert_multivectors =  list(colbert_model.embed(docs))

In [107]:
len(sparse_embeds), len(movies)

(9828, 9837)

In [134]:
from collections.abc import Mapping

from fastembed.common.types import NumpyArray
from qdrant_client.models import PointStruct, SparseVector

points = []

for i, idx in enumerate(docs_idx):
    # sparse_vector = sparse_embeds[i].as_object()
    sparse_vector = SparseVector(
        indices=sparse_embeds[i].indices.tolist(),
        values=sparse_embeds[i].values.tolist(),
    )

    dense_vector = dense_embeds[i]

    colbert_vector = colbert_multivectors[i]

    points.append(
        PointStruct(
            id=i,
            vector={
                'dense': dense_vector.tolist(),
                'sparse': sparse_vector,
                'colbert': colbert_vector.tolist()
            },
            payload=movies[idx] if isinstance(movies[idx], dict) else None,
        )
    )

In [109]:
points[0]

PointStruct(id=0, vector={'dense': [-0.048425689829488354, -0.008168368705491133, -0.005203856195401335, 0.04849431859363731, 0.013430981327798184, -0.008887358780722112, 0.007198892954711904, -0.028043158308630627, -0.10140284421065039, -0.039789918680731236, -0.053552480761695075, 0.023359080804387672, -0.08867982954084512, 0.052852324482400875, 0.027764983223893924, 0.007214069281634399, -0.019372960222060514, 0.023431515085851156, -0.0032376296555276366, 0.03295785474527477, 0.054826423964953026, 0.0005315551774932464, 0.03299215319847227, -0.06910631662104738, -0.022405893583938206, -0.031056567714500067, -0.0017894698834974602, -0.01787632349864433, 0.0459046238849515, -0.020727490958950445, -0.007953098239154915, -0.013354057507169247, -0.09401714237468148, -0.03281562301223058, -0.034449593948489836, -0.06224223024017425, -0.048403750321440876, 0.14686434313283722, -0.02263572930687666, -0.007397778791955593, -0.0076684261697561535, 0.016565101813413505, -0.06515665858023931, -

In [137]:
client.upload_points(collection_name=collection_name, points=points)

print(f"Points for the {collection_name} in size of {len(points)} are uploaded")

Points for the recommendation_movie in size of 9828 are uploaded


In [132]:
from datetime import datetime

user_query = "premium user likes sci-fi action movies with strong hacker themes"

user_dense_vector = next(iter(dense_model.query_embed(user_query))).tolist()
sparse_vector_query = next(iter(sparse_model.query_embed(user_query)))
user_sparse_vector = SparseVector(
    indices=sparse_vector_query.indices.tolist(),
    values=sparse_vector_query.values.tolist()
)
user_colber_vector = next(iter(colbert_model.query_embed(user_query)))



In [138]:
from datetime import datetime

from qdrant_client.models import (
    DatetimeRange,
    FieldCondition,
    Filter,
    Range,
)

global_filters= Filter(
    must=[
        FieldCondition(
            key="vote_average", range=Range(gte=7),
        ),
        FieldCondition(
            key="release_date", range=DatetimeRange(gte=datetime(2020, 1, 1)),
        ),
        FieldCondition(
            key="popularity", range=Range(gte=500),
        ),
    ]
)

In [139]:
from qdrant_client.models import Fusion, FusionQuery, Prefetch


hybrid_query = [
    Prefetch(query=user_dense_vector, using='dense'),
    Prefetch(query=user_sparse_vector, using='sparse'),
]

fusion_query = Prefetch(
    prefetch=hybrid_query, 
    query=FusionQuery(fusion=Fusion.RRF),
    limit=100
)

In [140]:
# Executre universal reranking

print(global_filters)

response = client.query_points(
    collection_name=collection_name,
    prefetch=fusion_query,
    query=user_colber_vector,
    query_filter=global_filters,
    using='colbert',
    limit=10,
    with_payload=True
)

should=None min_should=None must=[FieldCondition(key='vote_average', match=None, range=Range(lt=None, gt=None, gte=7.0, lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='release_date', match=None, range=DatetimeRange(lt=None, gt=None, gte=datetime.datetime(2020, 1, 1, 0, 0), lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None), FieldCondition(key='popularity', match=None, range=Range(lt=None, gt=None, gte=500.0, lte=None), geo_bounding_box=None, geo_radius=None, geo_polygon=None, values_count=None, is_empty=None, is_null=None)] must_not=None


In [141]:
for hit in response.points or []:
    print(hit.payload)

{'release_date': '2022-01-14T00:00:00', 'title': 'The House', 'overview': 'Across different eras, a poor family, an anxious developer and a fed-up landlady become tied to the same mysterious house in this animated dark comedy.', 'popularity': 551.65, 'vote_count': 432, 'vote_average': 7.1, 'original_language': 'en', 'genre': ['animation', 'drama', 'comedy', 'horror'], 'poster_url': 'https://image.tmdb.org/t/p/original/iZjMFSKCrleKolC1gYcz5Rs8bk1.jpg'}
{'release_date': '2020-10-16T00:00:00', 'title': 'Demon Slayer -Kimetsu no Yaiba- The Movie: Mugen Train', 'overview': "Tanjirō Kamado, joined with Inosuke Hashibira, a boy raised by boars who wears a boar's head, and Zenitsu Agatsuma, a scared boy who reveals his true power when he sleeps, boards the Infinity Train on a new mission with the Fire Hashira, Kyōjurō Rengoku, to defeat a demon who has been tormenting the people and killing the demon slayers who oppose it!", 'popularity': 845.992, 'vote_count': 2224, 'vote_average': 8.4, 'orig

In [150]:
from datetime import timedelta

from pydantic import BaseModel, Field
from qdrant_client.models import Condition, MatchValue

class UserProfile(BaseModel):
    liked_titles: list[str] = Field(default_factory=list)
    prefered_genres: list[str] = Field(default_factory=list)
    query: str


class UserPreference(BaseModel):
    released_within_days: int | None
    min_rating: float | None
    genre: str | None

def build_recommender_filter(_: UserProfile, user_preference: UserPreference):
    field_conditions: list[Condition] = []

    if user_preference.genre:
        field_conditions.append(
            FieldCondition(
                key='genre',
                match=MatchValue(value=user_preference.genre)
            )
        )

    if user_preference.min_rating:
        field_conditions.append(
            FieldCondition(
                key='vote_average',
                range=Range(gte=user_preference.min_rating)
            )
        )
    if user_preference.released_within_days:
        field_conditions.append(
            FieldCondition(
                key='release_date',
                range=DatetimeRange(gte=datetime.utcnow() - timedelta(days=user_preference.released_within_days))
            )
        )

    return Filter(must=field_conditions) if len(field_conditions) > 0 else None

In [153]:
def get_recommendations(user_profile: UserProfile, user_preference: UserPreference | None = None, limit=10):
    if user_preference is None:
        user_preference = UserPreference(released_within_days=None, min_rating=None, genre=None)
    
    user_dense_vector = next(iter(dense_model.query_embed(user_profile.query))).tolist()
    sparse_vector_query = next(iter(sparse_model.query_embed(user_profile.query)))
    user_sparse_vector = SparseVector(
        indices=sparse_vector_query.indices.tolist(),
        values=sparse_vector_query.values.tolist()
    )
    user_colbert_vector = next(iter(colbert_model.query_embed(user_profile.query)))


    global_filter = build_recommender_filter(user_profile, user_preference)

    hybrid_query = [
        Prefetch(query=user_dense_vector, using='dense', limit=100),
        Prefetch(query=user_sparse_vector, using='sparse', limit=100),
    ]

    fusion_query = Prefetch(
        prefetch=hybrid_query, 
        query=FusionQuery(fusion=Fusion.DBSF),
        limit=100
    )


    response = client.query_points(
        collection_name,
        prefetch=fusion_query,
        query_filter=global_filter,
        query=user_colbert_vector,
        using='colbert',
        limit=limit,
        with_payload=True,
    )

    return [
        {
            "title": point.payload['title'],
            "description": point.payload['overview'],
            "score": point.score,
            "metadata": {
                k: v
                for k, v in point.payload.items()
                if k not in ["title", "description"]
            },
        }
        for point in response.points if point.payload
    ]






In [ ]:
# Test the recommendation service
# user_profile = {
#     "liked_titles": ["The Matrix", "Blade Runner"],
#     "preferred_genres": ["sci-fi", "action"],
#     "segment": "premium",
#     "query": "highly rated cyberpunk movies",
# }

user_profile = UserProfile(
    liked_titles=["The Matrix", "Blade Runner"],
    prefered_genres=["sci-fi", "action"],
    query="highly rated cyberpunk movies"
)

user_preference = UserPreference(
    released_within_days=365*30,
    min_rating=7.7,
    genre='action'
)


# RRF results

recommendations = get_recommendations(
    user_profile,
    user_preference=user_preference,
    limit=10,
)

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['title']} (Score: {rec['score']:.3f})")

1. Sunny Side Battle! (Score: 11.339)
2. V.I.P. (Score: 10.892)
3. Fabricated City (Score: 10.756)
4. New Initial D the Movie - Legend 1: Awakening (Score: 10.430)
5. Primal: Tales of Savagery (Score: 10.335)
6. The Matrix (Score: 10.247)
7. Sword Art Online the Movie -Progressive- Aria of a Starless Night (Score: 9.805)
8. Teen Titans: Trouble in Tokyo (Score: 9.663)
9. Kill Bill: The Whole Bloody Affair (Score: 9.647)
10. Big Hero 6 (Score: 9.514)


/var/folders/pq/gsx7ryn905s1bypf32zs4k980000gn/T/ipykernel_83978/389935165.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  range=DatetimeRange(gte=datetime.utcnow() - timedelta(days=user_preference.released_within_days))


In [ ]:
# DBRF results
recommendations = get_recommendations(
    user_profile,
    user_preference=user_preference,
    limit=10,
)

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec['title']} (Score: {rec['score']:.3f})")

1. Sunny Side Battle! (Score: 11.339)
2. V.I.P. (Score: 10.892)
3. Fabricated City (Score: 10.756)
4. New Initial D the Movie - Legend 1: Awakening (Score: 10.430)
5. Primal: Tales of Savagery (Score: 10.335)
6. The Matrix (Score: 10.247)
7. Sword Art Online the Movie -Progressive- Aria of a Starless Night (Score: 9.805)
8. Teen Titans: Trouble in Tokyo (Score: 9.663)
9. Kill Bill: The Whole Bloody Affair (Score: 9.647)
10. Big Hero 6 (Score: 9.514)


/var/folders/pq/gsx7ryn905s1bypf32zs4k980000gn/T/ipykernel_83978/389935165.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  range=DatetimeRange(gte=datetime.utcnow() - timedelta(days=user_preference.released_within_days))


## [Day 5] Recommendations with the Universal Query API

**High-Level Summary**
- **Domain:** "I built recommendations for movies (TMDB-style dataset)"
- **Key Result:** "DBSF (dense + sparse) + ColBERT MaxSim rerank delivered a coherent cyberpunk/action top-10, with The Matrix breaking into rank #6 purely on semantic signal."

---

**Reproducibility**
- **Collection:** `recommendation_movie`
- **Models:** dense=[`sentence-transformers/all-MiniLM-L6-v2`, 384d], sparse=[`prithivida/Splade_PP_en_v1` / SPLADE++], colbert=[`colbert-ir/colbertv2.0`, 128d]
- **Dataset:** 9828 items from `Pablinho/movies-dataset` (9837 raw − 9 dropped for null `overview`) (snapshot: 2026-06-12)

---

**Settings (today)**
- **Fusion:** DBSF, prefetch `limit=100` per leg (dense + sparse)
- **Reranker:** ColBERT MaxSim, `top-k=10`
- **Filters:** `genre=action`, `vote_average ≥ 7.7`, `release_date ≥ now − 30 years (365×30 days)`
- **Filter propagation:** applied at outer `query_points` level; propagated by Qdrant to all prefetch stages

---

**Results (demo query: "highly rated cyberpunk movies" | liked: The Matrix, Blade Runner)**
- **Top picks:**
  1. Sunny Side Battle! — 11.339
  2. V.I.P. — 10.892
  3. Fabricated City — 10.756
  4. New Initial D the Movie - Legend 1: Awakening — 10.430
  5. Primal: Tales of Savagery — 10.335
  6. The Matrix — 10.247
  7. Sword Art Online the Movie -Progressive- Aria of a Starless Night — 9.805
  8. Teen Titans: Trouble in Tokyo — 9.663
  9. Kill Bill: The Whole Bloody Affair — 9.647
  10. Big Hero 6 — 9.514
- **Why these won:** Fabricated City and The Matrix hit on direct "hacker/cyberpunk" token alignment via SPLADE; SAO and Big Hero 6 win on dense sci-fi/tech semantics; V.I.P. and Kill Bill ride the `genre=action` filter + strong ColBERT token overlap on action sequences
- **Latency:** not yet instrumented
- **Dropped by rules:** films with `vote_average < 7.7`, non-action genre tags, or `release_date` before ~1996

---

**Surprise**
- "Kill Bill: The Whole Bloody Affair (#9) has nothing to do with hacking or cyberpunk, yet it surfaces — it clears all quality filters easily (highly rated Tarantino + genre=action), and ColBERT finds enough token overlap on action sequences to keep it in the reranked pool. A reminder that rating signal can override thematic relevance when the query has broad genre terms."

---

**Next step**
- A/B test RRF vs DBSF properly — currently both cells call the same `get_recommendations` (which hardcodes DBSF), so results are identical; wire in a `fusion` parameter to actually compare
- Experiment with more precise query phrasing — separate genre terms from user-tier signals ("cyberpunk hacker" vs "highly rated")
- Feed the user profile (`liked_titles`, `prefered_genres`) directly into the prefetch queries to bias retrieval before fusion, rather than relying solely on free-text query